In [1]:
# ============================================================
# SetStop-KGQA Notebook 05
# WebQSP Answer-Node Recognizer Training
# Complete single-cell Google Colab code
# ============================================================

!pip -q install -U \
    transformers \
    datasets \
    accelerate \
    scikit-learn \
    sentencepiece

# ============================================================
# 1. IMPORTS
# ============================================================

import collections
import gc
import hashlib
import inspect
import json
import math
import random
import re
import time

from pathlib import Path

import numpy as np
import torch
import transformers

from datasets import Dataset

from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    brier_score_loss,
    classification_report,
    confusion_matrix,
    f1_score,
    precision_recall_fscore_support,
)

from transformers import (
    AutoModelForSequenceClassification,
    AutoTokenizer,
    DataCollatorWithPadding,
    EarlyStoppingCallback,
    Trainer,
    TrainingArguments,
    set_seed,
)

from google.colab import drive


# ============================================================
# 2. CLEAN RUNTIME AND MOUNT DRIVE
# ============================================================

for variable_name in [
    "model",
    "trainer",
    "tokenizer",
]:
    if variable_name in globals():
        del globals()[variable_name]

gc.collect()

if torch.cuda.is_available():
    torch.cuda.empty_cache()

drive.mount("/content/drive")


# ============================================================
# 3. CONFIGURATION
# ============================================================

SEED = 42

set_seed(SEED)
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

ROOT = Path(
    "/content/drive/MyDrive/SetStop_KGQA"
)

DATA_DIR = (
    ROOT
    / "data/extracted/webqsp"
)

CACHE_ROOT = (
    ROOT
    / "data/answer_recognizer_cache_v1"
)

MODEL_ROOT = (
    ROOT
    / "models/answer_node_minilm_v1"
)

OUTPUT_ROOT = (
    ROOT
    / "outputs/answer_recognizer_v1"
)

CHECKPOINT_ROOT = (
    OUTPUT_ROOT
    / "checkpoints"
)

for directory in [
    CACHE_ROOT,
    MODEL_ROOT,
    OUTPUT_ROOT,
    CHECKPOINT_ROOT,
]:
    directory.mkdir(
        parents=True,
        exist_ok=True
    )

MODEL_NAME = (
    "microsoft/MiniLM-L12-H384-uncased"
)

MAX_HOPS = 3
MAX_ANSWERS = 50
MAX_SEQUENCE_LENGTH = 192

# Training uses three hard/medium negatives
# for every reachable positive answer.
TRAIN_NEGATIVES_PER_POSITIVE = 3

# Development uses a wider negative distribution
# for more realistic threshold calibration.
DEV_NEGATIVES_PER_POSITIVE = 10
MIN_DEV_NEGATIVES_PER_QUESTION = 50
MAX_DEV_NEGATIVES_PER_QUESTION = 400

TRAIN_CACHE = (
    CACHE_ROOT
    / "answer_node_train_v1.jsonl"
)

DEV_CACHE = (
    CACHE_ROOT
    / "answer_node_dev_v1.jsonl"
)

print("=" * 90)
print("WEBQSP ANSWER-NODE RECOGNIZER")
print("=" * 90)
print("Transformers:", transformers.__version__)
print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

assert torch.cuda.is_available(), (
    "Select a T4 GPU runtime."
)

GPU_NAME = torch.cuda.get_device_name(0)

print("GPU:", GPU_NAME)
print("Model:", MODEL_NAME)
print("=" * 90)


# ============================================================
# 4. DATA PARSING
# ============================================================

def read_vocabulary(path):
    with open(
        path,
        encoding="utf-8"
    ) as file:
        return [
            line.rstrip("\n")
            for line in file
        ]


ENTITY_NAMES = read_vocabulary(
    DATA_DIR / "entities.txt"
)

RELATION_NAMES = read_vocabulary(
    DATA_DIR / "relations.txt"
)

ENTITY_TO_ID = {
    name: index
    for index, name
    in enumerate(ENTITY_NAMES)
}

RELATION_TO_ID = {
    name: index
    for index, name
    in enumerate(RELATION_NAMES)
}


def iter_jsonl(path):
    with open(
        path,
        encoding="utf-8"
    ) as file:

        for line_number, line in enumerate(
            file,
            start=1
        ):
            if not line.strip():
                continue

            try:
                yield json.loads(line)

            except Exception as error:
                raise ValueError(
                    f"Invalid JSON at "
                    f"{path}:{line_number}"
                ) from error


def write_jsonl(rows, path):
    with open(
        path,
        "w",
        encoding="utf-8"
    ) as file:

        for row in rows:
            file.write(
                json.dumps(
                    row,
                    ensure_ascii=False
                )
                + "\n"
            )


def entity_index(value):
    if isinstance(value, (int, np.integer)):
        return int(value)

    if isinstance(value, str):
        return ENTITY_TO_ID.get(value)

    if isinstance(value, dict):
        for key in ["kb_id", "id", "entity"]:
            if key in value:
                return entity_index(
                    value[key]
                )

    return None


def relation_index(value):
    if isinstance(value, (int, np.integer)):
        return int(value)

    if isinstance(value, str):
        return RELATION_TO_ID.get(value)

    return None


def get_question(example):
    for key in ["question", "text", "query"]:
        value = example.get(key)

        if isinstance(value, str):
            return value.strip()

    raise KeyError(
        "Question field was not found"
    )


def get_topic_ids(example):
    raw_entities = example.get(
        "entities",
        example.get("topic_entities", [])
    )

    entity_ids = {
        entity_index(value)
        for value in raw_entities
    }

    return {
        value
        for value in entity_ids
        if value is not None
    }


def get_gold_ids(example):
    entity_ids = {
        entity_index(value)
        for value in example.get(
            "answers",
            []
        )
    }

    return {
        value
        for value in entity_ids
        if value is not None
    }


def get_graph(example):
    subgraph = example.get(
        "subgraph",
        {}
    )

    graph_entities = {
        entity_index(value)
        for value in subgraph.get(
            "entities",
            []
        )
    }

    graph_entities.discard(None)

    raw_edges = subgraph.get(
        "tuples",
        subgraph.get("triples", [])
    )

    edges = []

    for row in raw_edges:
        if isinstance(row, dict):
            subject = entity_index(
                row.get(
                    "subject",
                    row.get("s")
                )
            )

            relation = relation_index(
                row.get(
                    "relation",
                    row.get("r")
                )
            )

            object_entity = entity_index(
                row.get(
                    "object",
                    row.get("o")
                )
            )

        else:
            if len(row) != 3:
                continue

            subject = entity_index(row[0])
            relation = relation_index(row[1])
            object_entity = entity_index(
                row[2]
            )

        if (
            subject is None
            or relation is None
            or object_entity is None
        ):
            continue

        subject = int(subject)
        relation = int(relation)
        object_entity = int(object_entity)

        edges.append(
            (
                subject,
                relation,
                object_entity,
            )
        )

        graph_entities.add(subject)
        graph_entities.add(object_entity)

    return graph_entities, edges


def get_question_id(
    split,
    example,
    line_number
):
    for key in [
        "id",
        "qid",
        "question_id",
    ]:
        if key in example:
            return str(example[key])

    digest = hashlib.sha1(
        get_question(example).encode(
            "utf-8"
        )
    ).hexdigest()[:12]

    return (
        f"{split}_{line_number}_{digest}"
    )


def relation_name(relation_id):
    if 0 <= relation_id < len(
        RELATION_NAMES
    ):
        return RELATION_NAMES[
            relation_id
        ]

    return f"relation_{relation_id}"


# ============================================================
# 5. GRAPH PATH EXTRACTION
# ============================================================

def build_grouped_adjacency(edges):
    temporary = collections.defaultdict(
        lambda: collections.defaultdict(set)
    )

    for subject, relation, object_entity in edges:
        temporary[subject][relation].add(
            object_entity
        )

    adjacency = {}

    for source, relation_map in temporary.items():
        adjacency[source] = {
            relation: sorted(targets)
            for relation, targets
            in relation_map.items()
        }

    return adjacency


def extract_shortest_path_records(
    topic_entities,
    adjacency
):
    """
    Find one deterministic shortest relation path from
    the topic entities to each reachable entity.

    Entity IDs are retained for labels and evaluation but
    are not inserted into model text.
    """

    records = {}

    queue = collections.deque()

    for topic in sorted(topic_entities):
        queue.append(
            (
                topic,
                0,
                [],
            )
        )

    best_depth = {
        topic: 0
        for topic in topic_entities
    }

    while queue:
        source, depth, source_path = (
            queue.popleft()
        )

        if depth >= MAX_HOPS:
            continue

        relation_map = adjacency.get(
            source,
            {}
        )

        for relation in sorted(relation_map):
            targets = relation_map[relation]
            target_depth = depth + 1
            target_path = (
                source_path + [relation]
            )

            for target in targets:
                old_depth = best_depth.get(
                    target
                )

                if (
                    old_depth is None
                    or target_depth < old_depth
                ):
                    best_depth[target] = (
                        target_depth
                    )

                    records[target] = {
                        "node_id": target,
                        "depth": target_depth,
                        "path_relations":
                            target_path,
                        "incoming_relation":
                            relation,
                        "final_branch_size":
                            len(targets),
                        "outgoing_relation_count":
                            len(
                                adjacency.get(
                                    target,
                                    {}
                                )
                            ),
                    }

                    queue.append(
                        (
                            target,
                            target_depth,
                            target_path,
                        )
                    )

    return records


def make_model_text(
    question,
    record
):
    path_text = " -> ".join(
        relation_name(relation)
        for relation
        in record["path_relations"]
    )

    return (
        f"Question: {question}\n"
        f"Candidate relation path: {path_text}\n"
        f"Candidate depth: {record['depth']}\n"
        f"Incoming relation: "
        f"{relation_name(record['incoming_relation'])}\n"
        f"Entities returned by final relation: "
        f"{record['final_branch_size']}\n"
        f"Candidate outgoing relation count: "
        f"{record['outgoing_relation_count']}"
    )


# ============================================================
# 6. HARD-NEGATIVE SELECTION
# ============================================================

def select_training_negatives(
    positive_records,
    negative_records,
    target_count,
    rng
):
    if not negative_records:
        return []

    positive_signatures = {
        (
            record["depth"],
            record["incoming_relation"],
        )
        for record in positive_records
    }

    positive_depths = {
        record["depth"]
        for record in positive_records
    }

    hard = [
        record
        for record in negative_records
        if (
            record["depth"],
            record["incoming_relation"],
        ) in positive_signatures
    ]

    medium = [
        record
        for record in negative_records
        if (
            record not in hard
            and record["depth"]
            in positive_depths
        )
    ]

    easy = [
        record
        for record in negative_records
        if (
            record not in hard
            and record not in medium
        )
    ]

    rng.shuffle(hard)
    rng.shuffle(medium)
    rng.shuffle(easy)

    selected = []

    hard_target = math.ceil(
        target_count * 0.5
    )

    medium_target = math.ceil(
        target_count * 0.3
    )

    selected.extend(
        hard[:hard_target]
    )

    remaining_medium = (
        target_count - len(selected)
    )

    selected.extend(
        medium[
            :min(
                medium_target,
                remaining_medium
            )
        ]
    )

    selected_ids = {
        record["node_id"]
        for record in selected
    }

    remaining_pool = [
        record
        for record in (
            hard + medium + easy
        )
        if record["node_id"]
        not in selected_ids
    ]

    selected.extend(
        remaining_pool[
            :target_count - len(selected)
        ]
    )

    return selected[:target_count]


# ============================================================
# 7. BUILD ANSWER-RECOGNITION DATA
# ============================================================

def build_split_dataset(
    split,
    negative_ratio,
    development_mode=False
):
    source_path = (
        DATA_DIR
        / f"{split}_simple.json"
    )

    rows = []

    statistics = {
        "questions_total": 0,
        "questions_usable": 0,
        "questions_with_reachable_answers": 0,
        "gold_answers_total": 0,
        "gold_answers_reachable": 0,
        "positive_examples": 0,
        "negative_examples": 0,
    }

    for line_number, example in enumerate(
        iter_jsonl(source_path),
        start=1
    ):
        statistics["questions_total"] += 1

        question = get_question(example)
        topics = get_topic_ids(example)
        gold_answers = get_gold_ids(example)

        graph_nodes, graph_edges = (
            get_graph(example)
        )

        usable = (
            2 <= len(gold_answers)
            <= MAX_ANSWERS
            and gold_answers.issubset(
                graph_nodes
            )
            and bool(topics)
            and bool(graph_edges)
        )

        if not usable:
            continue

        statistics[
            "questions_usable"
        ] += 1

        statistics[
            "gold_answers_total"
        ] += len(gold_answers)

        qid = get_question_id(
            split,
            example,
            line_number
        )

        adjacency = build_grouped_adjacency(
            graph_edges
        )

        records = (
            extract_shortest_path_records(
                topic_entities=topics,
                adjacency=adjacency,
            )
        )

        reachable_gold = (
            gold_answers.intersection(
                records.keys()
            )
        )

        statistics[
            "gold_answers_reachable"
        ] += len(reachable_gold)

        if not reachable_gold:
            continue

        statistics[
            "questions_with_reachable_answers"
        ] += 1

        positive_records = [
            records[node_id]
            for node_id in reachable_gold
        ]

        negative_records = [
            record
            for node_id, record
            in records.items()
            if node_id not in gold_answers
        ]

        rng = random.Random(
            SEED
            + int(
                hashlib.sha1(
                    qid.encode("utf-8")
                ).hexdigest()[:8],
                16
            )
        )

        if development_mode:
            target_negative_count = min(
                MAX_DEV_NEGATIVES_PER_QUESTION,
                max(
                    MIN_DEV_NEGATIVES_PER_QUESTION,
                    len(positive_records)
                    * negative_ratio,
                ),
            )
        else:
            target_negative_count = (
                len(positive_records)
                * negative_ratio
            )

        selected_negatives = (
            select_training_negatives(
                positive_records=
                    positive_records,
                negative_records=
                    negative_records,
                target_count=
                    target_negative_count,
                rng=rng,
            )
        )

        for record in positive_records:
            rows.append({
                "qid": qid,
                "node_id":
                    record["node_id"],
                "text": make_model_text(
                    question,
                    record
                ),
                "label": 1,
                "depth": record["depth"],
                "hardness": "positive",
            })

        positive_signatures = {
            (
                record["depth"],
                record["incoming_relation"],
            )
            for record in positive_records
        }

        for record in selected_negatives:
            if (
                record["depth"],
                record["incoming_relation"],
            ) in positive_signatures:
                hardness = "hard"
            elif record["depth"] in {
                item["depth"]
                for item in positive_records
            }:
                hardness = "medium"
            else:
                hardness = "easy"

            rows.append({
                "qid": qid,
                "node_id":
                    record["node_id"],
                "text": make_model_text(
                    question,
                    record
                ),
                "label": 0,
                "depth": record["depth"],
                "hardness": hardness,
            })

        statistics[
            "positive_examples"
        ] += len(positive_records)

        statistics[
            "negative_examples"
        ] += len(selected_negatives)

        if (
            statistics["questions_usable"]
            % 200 == 0
        ):
            print(
                split,
                "usable questions:",
                statistics[
                    "questions_usable"
                ]
            )

    rng = random.Random(
        SEED if split == "train"
        else SEED + 1
    )

    rng.shuffle(rows)

    statistics[
        "total_examples"
    ] = len(rows)

    if statistics["gold_answers_total"]:
        statistics[
            "reachable_gold_fraction"
        ] = (
            statistics[
                "gold_answers_reachable"
            ]
            / statistics[
                "gold_answers_total"
            ]
        )
    else:
        statistics[
            "reachable_gold_fraction"
        ] = 0.0

    return rows, statistics


if TRAIN_CACHE.exists():
    print(
        "\nLoading cached answer-node "
        "training data..."
    )

    train_rows = list(
        iter_jsonl(TRAIN_CACHE)
    )

    train_statistics = {
        "loaded_from_cache": True,
        "total_examples":
            len(train_rows),
        "class_counts": dict(
            collections.Counter(
                row["label"]
                for row in train_rows
            )
        ),
    }

else:
    print(
        "\nBuilding answer-node "
        "training data..."
    )

    train_rows, train_statistics = (
        build_split_dataset(
            split="train",
            negative_ratio=
                TRAIN_NEGATIVES_PER_POSITIVE,
            development_mode=False,
        )
    )

    write_jsonl(
        train_rows,
        TRAIN_CACHE
    )


if DEV_CACHE.exists():
    print(
        "\nLoading cached answer-node "
        "development data..."
    )

    dev_rows = list(
        iter_jsonl(DEV_CACHE)
    )

    dev_statistics = {
        "loaded_from_cache": True,
        "total_examples":
            len(dev_rows),
        "class_counts": dict(
            collections.Counter(
                row["label"]
                for row in dev_rows
            )
        ),
    }

else:
    print(
        "\nBuilding answer-node "
        "development data..."
    )

    dev_rows, dev_statistics = (
        build_split_dataset(
            split="dev",
            negative_ratio=
                DEV_NEGATIVES_PER_POSITIVE,
            development_mode=True,
        )
    )

    write_jsonl(
        dev_rows,
        DEV_CACHE
    )


print("\nTraining-data statistics")
print(json.dumps(
    train_statistics,
    indent=2
))

print("\nDevelopment-data statistics")
print(json.dumps(
    dev_statistics,
    indent=2
))

assert train_rows, (
    "No training examples created"
)

assert dev_rows, (
    "No development examples created"
)

assert {
    row["qid"]
    for row in train_rows
}.isdisjoint({
    row["qid"]
    for row in dev_rows
}), (
    "Question leakage between "
    "training and development"
)


# ============================================================
# 8. DATASETS AND TOKENIZATION
# ============================================================

train_dataset = Dataset.from_list([
    {
        "text": row["text"],
        "labels": int(row["label"]),
    }
    for row in train_rows
])

dev_dataset = Dataset.from_list([
    {
        "text": row["text"],
        "labels": int(row["label"]),
    }
    for row in dev_rows
])

tokenizer = AutoTokenizer.from_pretrained(
    MODEL_NAME,
    use_fast=True,
)


def tokenize_batch(batch):
    return tokenizer(
        batch["text"],
        truncation=True,
        max_length=MAX_SEQUENCE_LENGTH,
        padding=False,
    )


tokenized_train = train_dataset.map(
    tokenize_batch,
    batched=True,
    remove_columns=["text"],
    desc="Tokenizing training data",
)

tokenized_dev = dev_dataset.map(
    tokenize_batch,
    batched=True,
    remove_columns=["text"],
    desc="Tokenizing development data",
)

data_collator = DataCollatorWithPadding(
    tokenizer=tokenizer,
    pad_to_multiple_of=8,
)


# ============================================================
# 9. MODEL
# ============================================================

model = (
    AutoModelForSequenceClassification
    .from_pretrained(
        MODEL_NAME,
        num_labels=2,
        id2label={
            0: "not_answer",
            1: "answer",
        },
        label2id={
            "not_answer": 0,
            "answer": 1,
        },
        dtype=torch.float32,
    )
)


# ============================================================
# 10. METRICS
# ============================================================

def extract_logits(predictions):
    if isinstance(predictions, tuple):
        return predictions[0]

    return predictions


def compute_metrics(
    evaluation_prediction
):
    logits = extract_logits(
        evaluation_prediction.predictions
    )

    labels = (
        evaluation_prediction.label_ids
    )

    predictions = np.argmax(
        logits,
        axis=1
    )

    return {
        "accuracy": float(
            accuracy_score(
                labels,
                predictions
            )
        ),
        "balanced_accuracy": float(
            balanced_accuracy_score(
                labels,
                predictions
            )
        ),
        "macro_f1": float(
            f1_score(
                labels,
                predictions,
                average="macro",
                zero_division=0,
            )
        ),
        "answer_f1": float(
            f1_score(
                labels,
                predictions,
                pos_label=1,
                zero_division=0,
            )
        ),
    }


# ============================================================
# 11. VERSION-COMPATIBLE TRAINING ARGUMENTS
# ============================================================

steps_per_epoch = math.ceil(
    len(train_dataset) / 32
)

warmup_steps = max(
    25,
    int(steps_per_epoch * 0.05)
)

argument_parameters = set(
    inspect.signature(
        TrainingArguments.__init__
    ).parameters
)

argument_values = {
    "output_dir":
        str(CHECKPOINT_ROOT),

    "num_train_epochs": 2,

    "per_device_train_batch_size": 32,
    "per_device_eval_batch_size": 64,

    "gradient_accumulation_steps": 1,

    "learning_rate": 3e-5,
    "weight_decay": 0.01,
    "warmup_steps": warmup_steps,

    "lr_scheduler_type": "linear",

    "fp16": True,
    "bf16": False,

    "logging_strategy": "steps",
    "logging_steps": 25,

    "save_strategy": "epoch",
    "save_total_limit": 2,

    "load_best_model_at_end": True,
    "metric_for_best_model": "answer_f1",
    "greater_is_better": True,

    "max_grad_norm": 1.0,
    "report_to": "none",

    "remove_unused_columns": True,

    "seed": SEED,
    "data_seed": SEED,

    "dataloader_num_workers": 2,
}

if "eval_strategy" in argument_parameters:
    argument_values[
        "eval_strategy"
    ] = "epoch"

elif (
    "evaluation_strategy"
    in argument_parameters
):
    argument_values[
        "evaluation_strategy"
    ] = "epoch"

else:
    raise RuntimeError(
        "No supported evaluation-strategy "
        "argument was found."
    )

supported_arguments = {
    key: value
    for key, value
    in argument_values.items()
    if key in argument_parameters
}

training_arguments = TrainingArguments(
    **supported_arguments
)


# ============================================================
# 12. VERSION-COMPATIBLE TRAINER
# ============================================================

trainer_parameters = set(
    inspect.signature(
        Trainer.__init__
    ).parameters
)

trainer_values = {
    "model": model,
    "args": training_arguments,
    "train_dataset": tokenized_train,
    "eval_dataset": tokenized_dev,
    "data_collator": data_collator,
    "compute_metrics": compute_metrics,
    "callbacks": [
        EarlyStoppingCallback(
            early_stopping_patience=2
        )
    ],
}

if "processing_class" in trainer_parameters:
    trainer_values[
        "processing_class"
    ] = tokenizer

elif "tokenizer" in trainer_parameters:
    trainer_values[
        "tokenizer"
    ] = tokenizer


trainer = Trainer(
    **trainer_values
)


# ============================================================
# 13. TRAIN
# ============================================================

print("\nStarting answer recognizer training...")

training_start = time.time()

training_result = trainer.train()

training_minutes = (
    time.time() - training_start
) / 60

trainer.save_model(
    str(MODEL_ROOT)
)

tokenizer.save_pretrained(
    str(MODEL_ROOT)
)

print(
    "Training completed in",
    round(training_minutes, 2),
    "minutes"
)


# ============================================================
# 14. DEVELOPMENT PREDICTIONS
# ============================================================

prediction_output = trainer.predict(
    tokenized_dev
)

logits = extract_logits(
    prediction_output.predictions
)

labels = np.asarray(
    prediction_output.label_ids
)

shifted_logits = (
    logits
    - np.max(
        logits,
        axis=1,
        keepdims=True
    )
)

exponentials = np.exp(
    shifted_logits
)

probabilities = (
    exponentials
    / np.sum(
        exponentials,
        axis=1,
        keepdims=True
    )
)

answer_probabilities = (
    probabilities[:, 1]
)


# ============================================================
# 15. THRESHOLD CALIBRATION
# ============================================================

def evaluate_threshold(threshold):
    predictions = (
        answer_probabilities
        >= threshold
    ).astype(np.int64)

    precision, recall, answer_f1, _ = (
        precision_recall_fscore_support(
            labels,
            predictions,
            average="binary",
            zero_division=0,
        )
    )

    return {
        "threshold": float(threshold),
        "precision": float(precision),
        "recall": float(recall),
        "answer_f1": float(answer_f1),
        "accuracy": float(
            accuracy_score(
                labels,
                predictions
            )
        ),
        "balanced_accuracy": float(
            balanced_accuracy_score(
                labels,
                predictions
            )
        ),
    }


threshold_results = [
    evaluate_threshold(threshold)
    for threshold in np.linspace(
        0.05,
        0.95,
        91
    )
]

best_threshold_result = max(
    threshold_results,
    key=lambda result: (
        result["answer_f1"],
        result["precision"],
        result["recall"],
    )
)

selected_threshold = (
    best_threshold_result["threshold"]
)

final_predictions = (
    answer_probabilities
    >= selected_threshold
).astype(np.int64)


# ============================================================
# 16. FINAL DEVELOPMENT METRICS
# ============================================================

development_metrics = {
    "selected_threshold":
        selected_threshold,

    "number_of_examples":
        int(len(labels)),

    "positive_examples":
        int(np.sum(labels == 1)),

    "negative_examples":
        int(np.sum(labels == 0)),

    "accuracy": float(
        accuracy_score(
            labels,
            final_predictions
        )
    ),

    "balanced_accuracy": float(
        balanced_accuracy_score(
            labels,
            final_predictions
        )
    ),

    "macro_f1": float(
        f1_score(
            labels,
            final_predictions,
            average="macro",
            zero_division=0,
        )
    ),

    "answer_f1": float(
        f1_score(
            labels,
            final_predictions,
            pos_label=1,
            zero_division=0,
        )
    ),

    "brier_score": float(
        brier_score_loss(
            labels,
            answer_probabilities
        )
    ),

    "classification_report":
        classification_report(
            labels,
            final_predictions,
            target_names=[
                "not_answer",
                "answer",
            ],
            output_dict=True,
            zero_division=0,
        ),

    "confusion_matrix":
        confusion_matrix(
            labels,
            final_predictions,
            labels=[0, 1],
        ).tolist(),
}


# ============================================================
# 17. SAVE OUTPUTS
# ============================================================

training_summary = {
    "model": MODEL_NAME,
    "saved_model": str(MODEL_ROOT),
    "gpu": GPU_NAME,
    "transformers_version":
        transformers.__version__,
    "pytorch_version":
        torch.__version__,
    "training_minutes":
        float(training_minutes),
    "train_examples":
        len(train_rows),
    "development_examples":
        len(dev_rows),
    "train_statistics":
        train_statistics,
    "development_statistics":
        dev_statistics,
    "selected_threshold":
        selected_threshold,
    "development_answer_f1":
        development_metrics[
            "answer_f1"
        ],
    "development_macro_f1":
        development_metrics[
            "macro_f1"
        ],
    "development_brier_score":
        development_metrics[
            "brier_score"
        ],
}

summary_path = (
    OUTPUT_ROOT
    / "answer_recognizer_summary_v1.json"
)

metrics_path = (
    OUTPUT_ROOT
    / "answer_recognizer_dev_metrics_v1.json"
)

threshold_path = (
    OUTPUT_ROOT
    / "answer_recognizer_threshold_v1.json"
)

summary_path.write_text(
    json.dumps(
        training_summary,
        indent=2
    ),
    encoding="utf-8",
)

metrics_path.write_text(
    json.dumps(
        development_metrics,
        indent=2
    ),
    encoding="utf-8",
)

threshold_path.write_text(
    json.dumps(
        {
            "selection_split":
                "development",
            "selected_threshold":
                selected_threshold,
            "selected_metrics":
                best_threshold_result,
            "all_threshold_results":
                threshold_results,
        },
        indent=2
    ),
    encoding="utf-8",
)


# ============================================================
# 18. FINAL OUTPUT
# ============================================================

print("\n" + "=" * 90)
print("ANSWER-NODE RECOGNIZER TRAINING COMPLETED")
print("=" * 90)

print(json.dumps(
    training_summary,
    indent=2
))

print("\nDevelopment metrics:")
print(json.dumps(
    development_metrics,
    indent=2
))

print("\nSaved model:")
print(MODEL_ROOT)

print("\nSaved outputs:")
print(summary_path)
print(metrics_path)
print(threshold_path)

print("=" * 90)

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.3/12.3 MB 41.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 559.1/559.1 kB 25.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 394.3/394.3 kB 30.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.1/9.1 MB 80.5 MB/s eta 0:00:00
Mounted at /content/drive
WEBQSP ANSWER-NODE RECOGNIZER
Transformers: 5.17.0
PyTorch: 2.11.0+cu128
CUDA available: True
GPU: Tesla T4
Model: microsoft/MiniLM-L12-H384-uncased

Building answer-node training data...
train usable questions: 200
train usable questions: 400
train usable questions: 600
train usable questions: 800
train usable questions: 1000
train usable questions: 1200

Building answer-node development data...

Training-data statistics
{
  "questions_total": 2848,
  "questions_usable": 1200,
  "questions_with_reachable_answers": 1200,
  "gold_answers_total": 8177,
  "gold_answers_reachable": 8175,
  "positive_examples": 8175,
  "negative_examples": 24525,
  "tot

/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_auth.py:138: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
  warnings.warn(f"\nError while fetching `HF_TOKEN` secret value from your vault: '{str(e)}'.")


config.json:   0%|          | 0.00/385 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/2.00 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Training completed in 2.48 minutes



ANSWER-NODE RECOGNIZER TRAINING COMPLETED
{
  "model": "microsoft/MiniLM-L12-H384-uncased",
  "saved_model": "/content/drive/MyDrive/SetStop_KGQA/models/answer_node_minilm_v1",
  "gpu": "Tesla T4",
  "transformers_version": "5.17.0",
  "pytorch_version": "2.11.0+cu128",
  "training_minutes": 2.4804245432217917,
  "train_examples": 32700,
  "development_examples": 9549,
  "train_statistics": {
    "questions_total": 2848,
    "questions_usable": 1200,
    "questions_with_reachable_answers": 1200,
    "gold_answers_total": 8177,
    "gold_answers_reachable": 8175,
    "positive_examples": 8175,
    "negative_examples": 24525,
    "total_examples": 32700,
    "reachable_gold_fraction": 0.9997554115201174
  },
  "development_statistics": {
    "questions_total": 250,
    "questions_usable": 118,
    "questions_with_reachable_answers": 118,
    "gold_answers_total": 719,
    "gold_answers_reachable": 719,
    "positive_examples": 719,
    "negative_examples": 8830,
    "total_examples": 95